# 	•	Combined data from Elgiganten and Komplett.
#	•	Added a “Source” column to distinguish them.

# This code normalizes the specs (CPU, RAM, etc.) across both datasets

In [28]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import numpy as np

# Load Excel files and keep only relevant columns
use_cols = ["Name", "Price", "CPU", "RAM", "Storage"]
elgiganten_df = pd.read_excel("../Step 1 - Scrape Elgiganten and Komplett laptop listings/elgiganten_laptops_updated.xlsx", usecols=use_cols)
komplett_df = pd.read_excel("../Step 1 - Scrape Elgiganten and Komplett laptop listings/komplett_laptops_updated.xlsx", usecols=use_cols)

# Add retailer labels
elgiganten_df["Retailer"] = "Elgiganten"
komplett_df["Retailer"] = "Komplett"

# Normalization functions
def normalize_price(price):
    if isinstance(price, str):
        price = price.replace(":-", "").replace(" ", "").replace("kr", "")
    try:
        return int(price)
    except:
        return None

def normalize_ram(ram):
    if isinstance(ram, str):
        match = re.search(r"(\d+)", ram.replace(" ", ""))
        return int(match.group(1)) if match else None
    return None

def normalize_storage(storage):
    if isinstance(storage, str):
        storage = storage.lower().replace(" ", "")
        match = re.search(r"(\d+(?:\.\d+)?)", storage)
        if "tb" in storage:
            return int(float(match.group(1)) * 1024) if match else None
        elif "gb" in storage:
            return int(match.group(1)) if match else None
    return None

# Apply normalization
elgiganten_df["Price"] = elgiganten_df["Price"].apply(normalize_price)
elgiganten_df["RAM_GB"] = elgiganten_df["RAM"].apply(normalize_ram)
elgiganten_df["Storage_GB"] = elgiganten_df["RAM"].apply(normalize_storage)  # storage is embedded in RAM

komplett_df["Price"] = komplett_df["Price"].apply(normalize_price)
komplett_df["RAM_GB"] = komplett_df["RAM"].apply(normalize_ram)
komplett_df["Storage_GB"] = komplett_df["Storage"].apply(normalize_storage)

# Combine into one DataFrame
df = pd.concat([elgiganten_df, komplett_df], ignore_index=True)

# Fill text columns
df["CPU"] = df["CPU"].fillna("unknown_cpu")

# Drop rows with missing price (critical for clustering)
df = df.dropna(subset=["Price"])

# Text features (CPU only)
df["TextFeatures"] = df["CPU"]
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df["TextFeatures"])

# Numeric subset: drop rows with missing numeric features
numeric_subset = df.dropna(subset=["RAM_GB", "Storage_GB"])
scaled_numeric = StandardScaler().fit_transform(numeric_subset[["RAM_GB", "Storage_GB", "Price"]].values)

# Match TF-IDF with same subset
tfidf_numeric_subset = tfidf_matrix[df.index.get_indexer(numeric_subset.index)]

# Combine numeric and text
combined_features = np.hstack([scaled_numeric, tfidf_numeric_subset.toarray()])

# Clustering
kmeans = KMeans(n_clusters=10, random_state=42)
numeric_subset["Cluster"] = kmeans.fit_predict(combined_features)

# Merge Cluster info back into full DataFrame
df = df.merge(numeric_subset[["Name", "Cluster"]], on="Name", how="left")

# Keep only clusters with both retailers
valid_clusters = df["Cluster"].dropna().unique()
valid_clusters = [cid for cid in valid_clusters if 
                  set(df[df["Cluster"] == cid]["Retailer"].unique()) == {"Elgiganten", "Komplett"}]

df_filtered = df[df["Cluster"].isin(valid_clusters)]

# Save result
df_filtered.to_csv("clustered_laptops_1.csv", index=False, encoding="utf-8-sig")
print("Done. Saved to clustered_laptops_1.csv")

Done. Saved to clustered_laptops_1.csv


/var/folders/mw/sdgy77fx2db5q5mt0f3kl8gr0000gn/T/ipykernel_15563/79579221.py:77: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  numeric_subset["Cluster"] = kmeans.fit_predict(combined_features)
